## STEP 1)Data Preparation & Model Loading
In this step I loaded a pre trained VGG16 model and changed the final classifier so that it could work with the 10 classes in the CIFAR-10 dataset I then moved the model to the available device such as the GPU and froze the feature extraction layers so that these pre trained features would not be changed during training I then created separate training and testing transformations including flipping cropping resizing converting the images to tensors and normalising them then finally I loaded the CIFAR-10 training and testing datasets split the training data in to training and validation sets and created data loaders to process the images in batches

In [1]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader, random_split
from PIL import Image
import requests
from io import BytesIO
import torch.nn as nn
import torch.optim as optim

# Load the pre-trained VGG16 model
model = models.vgg16(pretrained=True)

# Modify the classifier to fit CIFAR-10
model.classifier[6] = nn.Linear(4096, 10)

# Move the model to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Freeze the feature extractor layers
for param in model.features.parameters():
    param.requires_grad = False

# Define transformations
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

transform_test = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

# Load datasets
train_dataset = CIFAR10(root='./data', train=True, download=True, transform=transform_train)
test_dataset = CIFAR10(root='./data', train=False, download=True, transform=transform_test)

# Split the training data into training and validation sets
train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_dataset, val_dataset = random_split(train_dataset, [train_size, val_size])

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=4)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:02<00:00, 185MB/s] 
100%|██████████| 170M/170M [09:53<00:00, 287kB/s] 


## Step 2) Training

In this step I trained the VGG16 model using a cross entropy loss function and the SGD optimiser the optimiser updates the final classifier layer off the model so that it can learn to classify the 10 different CIFAR-10 classes the training is carried out for two epochs with the images processed in batches from the training data loader and during each batch the model makes predictions calculates the loss performs backpropagation and updates the model's weights after each epoch the model is tested using the validation dataset to calculate the validation accuracy and check how well it is learning

In [2]:
# Define the loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.classifier[6].parameters(), lr=0.001, momentum=0.9)

# Training loop
num_epochs = 2
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(train_loader)}")

    # Validation
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print(f"Validation Accuracy: {100 * correct / total}%")

Epoch 1/2, Loss: 0.8236129432678223
Validation Accuracy: 79.17%
Epoch 2/2, Loss: 0.6683351605892182
Validation Accuracy: 80.4%


## STEP 3) Data loading and checkpoints
In this step I tested the trained VGG16 model using the test dataset to measure how accurately it can classify unseen CIFAR-10 images the model was put into evaluation mode and torch.no_grad() was used so that gradients were not calculated during testing the predictions were compared with the correct labels to calculate the overall test accuracy I then saved the model's learned parameters as a checkpoint called vgg16_cifar10.pth finally I loaded the saved checkpoint back into the model and placed it into evaluation mode so that the trained model could be used again with out needing to retrain it

In [3]:
# Test the model
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Test Accuracy: {100 * correct / total}%")

# Save the model checkpoint
torch.save(model.state_dict(), 'vgg16_cifar10.pth')

# Load the model checkpoint
model.load_state_dict(torch.load('vgg16_cifar10.pth'))
model.eval()

Test Accuracy: 80.59%


VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1